In [1]:
pip install duckdb 


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import duckdb, os

f = "test_scans.json.zst"
print("Looking for:", os.path.abspath(f), "| exists:", os.path.exists(f))

Looking for: c:\Users\loyol\OneDrive\Documents\AU\Firmable-task\test_scans.json.zst | exists: True


In [4]:
structure = duckdb.sql(
    f"SELECT json_group_structure(json) FROM (SELECT * FROM read_ndjson_objects('{f}') LIMIT 2048)"
).fetchone()[0]
print(structure)

{"data":"VARCHAR","opts":{"vulns":["NULL"],"heartbleed":"VARCHAR","raw":"VARCHAR","data":"VARCHAR"},"port":"UBIGINT","hostnames":["VARCHAR"],"timestamp":"VARCHAR","domains":["VARCHAR"],"location":{"area_code":"NULL","city":"VARCHAR","country_code":"VARCHAR","country_name":"VARCHAR","latitude":"DOUBLE","longitude":"DOUBLE","region_code":"VARCHAR"},"org":"VARCHAR","isp":"VARCHAR","os":"VARCHAR","http":{"status":"UBIGINT","robots_hash":"HUGEINT","redirects":[{"host":"VARCHAR","data":"VARCHAR","location":"VARCHAR","html":"VARCHAR"}],"title_hash":"HUGEINT","robots":"VARCHAR","dom_hash":"HUGEINT","host":"VARCHAR","html_hash":"HUGEINT","securitytxt":"NULL","title":"VARCHAR","sitemap_hash":"UBIGINT","waf":"VARCHAR","server":"VARCHAR","headers_hash":"HUGEINT","html":"VARCHAR","location":"VARCHAR","components":{"HTTP/3":{"categories":["VARCHAR"]},"Cloudflare":{"categories":["VARCHAR"]},"HSTS":{"categories":["VARCHAR"]},"AngularJS":{"categories":["VARCHAR"]},"Amazon ELB":{"categories":["VARCHAR"]

In [11]:
q = f"""
SELECT
  count(*)                                                              AS total,
  count(*) FILTER (WHERE json_extract(json, '$.vulns') IS NOT NULL)     AS has_vulns,
  count(*) FILTER (WHERE json_extract(json, '$.ssl')   IS NOT NULL)     AS has_ssl,
  count(*) FILTER (WHERE json_array_length(json_extract(json, '$.hostnames')) > 0) AS has_hostnames,
  count(*) FILTER (WHERE json_extract(json, '$.org')   IS NOT NULL)     AS has_org
FROM read_ndjson_objects('{f}', ignore_errors=true, maximum_object_size=134217728)
"""
duckdb.sql(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total,has_vulns,has_ssl,has_hostnames,has_org
0,6124461,120359,582544,4106040,6124319


In [5]:
duckdb.sql(f"""
DESCRIBE SELECT * FROM read_json_auto('{f}', ignore_errors=true, maximum_object_size=134217728)
""").df()

,column_name,column_type,null,key,default,extra
0,data,VARCHAR,YES,None,None,None
1,opts,"MAP(VARCHAR, JSON)",YES,None,None,None
2,port,BIGINT,YES,None,None,None
3,hostnames,VARCHAR[],YES,None,None,None
4,timestamp,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
75,plex,STRUCT(machine_identifier VARCHAR),YES,None,None,None
76,stun,"STRUCT(server_ip VARCHAR, software VARCHAR)",YES,None,None,None
77,dahua_dvr_web,"STRUCT(web_version VARCHAR, plugin STRUCT(""ver...",YES,None,None,None
78,ethereum_rpc,"STRUCT(client VARCHAR, ""version"" VARCHAR, plat...",YES,None,None,None


In [6]:
import json

# 1. COLUMNS
print("="*30, "COLUMNS", "="*30)
duckdb.sql(f"DESCRIBE SELECT * FROM read_json_auto('{f}', ignore_errors=true, maximum_object_size=134217728)").show()

# 2. SAMPLE ROWS
print("="*30, "SAMPLE ROWS", "="*30)
for r in duckdb.sql(f"SELECT json FROM read_ndjson_objects('{f}', ignore_errors=true, maximum_object_size=134217728) LIMIT 3").fetchall():
    print(json.dumps(json.loads(r[0]), indent=2)[:2000], "\n" + "-"*80)

# 3. EVERY COLUMN + SAMPLE VALUES
print("="*30, "FIELDS + SAMPLE VALUES", "="*30)
keys = duckdb.sql(f"""
  SELECT DISTINCT unnest(json_keys(json)) AS k
  FROM (SELECT json FROM read_ndjson_objects('{f}', ignore_errors=true, maximum_object_size=134217728) LIMIT 5000)
  ORDER BY k
""").df()["k"]
for k in keys:
    vals = duckdb.sql(f"""
      SELECT json_extract_string(json, '$.{k}') AS s
      FROM read_ndjson_objects('{f}', ignore_errors=true, maximum_object_size=134217728)
      WHERE json_extract_string(json, '$.{k}') IS NOT NULL LIMIT 3
    """).fetchall()
    print(f"{k:20} -> {[ (v[0][:60] if v[0] else v[0]) for v in vals ]}")

============================== COLUMNS ==============================
┌───────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [5]:
duckdb.sql(f"""
SELECT
  json_extract_string(json, '$.hostnames')            AS hostnames,
  json_extract_string(json, '$.ssl.cert.subject.CN')  AS cert_cn
FROM read_ndjson_objects('{f}', ignore_errors=true, maximum_object_size=134217728)
WHERE json_extract_string(json, '$.hostnames') IS NOT NULL
  AND json_extract_string(json, '$.hostnames') != '[]'
LIMIT 15
""").df()

,hostnames,cert_cn
0,"[""escalade-climbing.com""]",escalade-climbing.com
1,"[""hughessnell.com""]",hughessnell.com
2,"[""110.43.149.34.bc.googleusercontent.com""]",None
3,"[""93.181.102.34.bc.googleusercontent.com""]",None
4,"[""inmotionhosting.com"",""ecbiz237.inmotionhosti...",*.inmotionhosting.com
5,"[""192.118.120.34.bc.googleusercontent.com""]",None
6,"[""www.clubkuma.com"",""acd89244c803f7181.awsglob...",clubkuma.com
7,"[""host-95-229-13-77.business.telecomitalia.it""]",usg_flex_100_FC22F4E3A74A
8,"[""84.159.244.35.bc.googleusercontent.com""]",None
9,"[""229.197.54.34.bc.googleusercontent.com""]",None


In [6]:
import duckdb

# providers we never treat as a company
PROVIDERS = ['googleusercontent','amazonaws','azure','cloudapp','telecomitalia',
             'inmotionhosting','managed-vps','compute.amazonaws','1e100',
             'your-server.de','contabo','ovh','digitalocean','akamai','cloudfront']

block = " AND ".join([f"host NOT LIKE '%{p}%'" for p in PROVIDERS])

q = f"""
WITH raw AS (
  SELECT
    lower(json_extract_string(json, '$.hostnames[0]')) AS host,
    lower(json_extract_string(json, '$.ssl.cert.subject.CN')) AS cn,
    json_extract_string(json, '$.location.country_name') AS country
  FROM read_ndjson_objects('{f}', ignore_errors=true, maximum_object_size=134217728)
  WHERE json_extract_string(json, '$.hostnames') IS NOT NULL
    AND json_extract_string(json, '$.hostnames') != '[]'
),
clean AS (
  SELECT
    -- prefer cert CN if it's a real domain, else the hostname
    coalesce(nullif(regexp_replace(cn, '^\\*\\.', ''), ''), host) AS domain,
    country
  FROM raw
  WHERE {block}
    AND host NOT SIMILAR TO '.*[0-9]+[.-][0-9]+.*'   -- kills IP-in-hostname junk
)
SELECT
  -- crude root-domain grab: last two labels
  regexp_extract(domain, '([^.]+\\.[^.]+)$', 1) AS company_domain,
  any_value(country) AS country,
  count(*) AS servers
FROM clean
WHERE domain IS NOT NULL AND domain != ''
GROUP BY 1
ORDER BY servers DESC
LIMIT 40
"""
duckdb.sql(q).df()

,company_domain,country,servers
0,t-ipconnect.de,Germany,61506
1,imperva.com,Singapore,51965
2,ne.jp,Japan,34767
3,walmart.com,United States,18625
4,nuro.jp,Japan,14180
5,,Japan,13738
6,bluehost.com,United States,13302
7,co.jp,Japan,9259
8,bizmw.com,Japan,9048
9,co.uk,United Kingdom,8528


In [8]:
pip install tldextract

  Using cached filelock-3.32.2-py3-none-any.whl.metadata (2.0 kB)
Using cached filelock-3.32.2-py3-none-any.whl (98 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# run once: pip install tldextract
import duckdb, tldextract

# pull raw hostnames + cert, minimal cleaning, into pandas
df = duckdb.sql(f"""
  SELECT
    lower(json_extract_string(json, '$.hostnames[0]'))        AS host,
    lower(json_extract_string(json, '$.ssl.cert.subject.CN')) AS cn,
    json_extract_string(json, '$.location.country_name')      AS country
  FROM read_ndjson_objects('{f}', ignore_errors=true, maximum_object_size=134217728)
  WHERE json_extract_string(json, '$.hostnames') != '[]'
""").df()

extract = tldextract.TLDExtract(suffix_list_urls=())  # offline, uses bundled list

def root(h):
    if not h: return None
    e = extract(h)
    if not e.domain or not e.suffix: return None
    return f"{e.domain}.{e.suffix}"

# prefer cert domain, else hostname; strip wildcard
def pick(cn, host):
    cand = (cn or "").lstrip("*.") or host
    return root(cand)

df["company"] = [pick(c, h) for c, h in zip(df["cn"], df["host"])]

# obvious provider suffixes to drop outright
BAD = {"googleusercontent.com","amazonaws.com","t-ipconnect.de","secureserver.net",
       "bluehost.com","hostgator.com","xserver.jp","dreamhostps.com","wixsite.com",
       "weebly.com","hostmonster.com","websitewelcome.com","dattaweb.com","kasserver.com",
       "webhostbox.net","coreserver.jp","managed-vps.net","inmotionhosting.com"}

g = (df[df["company"].notna() & ~df["company"].isin(BAD)]
       .groupby("company")
       .agg(servers=("company","size"), country=("country","first"))
       .reset_index()
       .sort_values("servers", ascending=False))

# providers hide as huge counts — show the mid-range where real companies live
print("=== suspiciously large (likely providers, review) ===")
print(g[g.servers > 500].head(20).to_string())
print("\n=== realistic company range ===")
print(g[(g.servers >= 2) & (g.servers <= 500)].head(40).to_string())

=== suspiciously large (likely providers, review) ===
                       company  servers         country
89499             incapdns.net  1088753       Singapore
89236              imperva.com    68892       Singapore
180519        telecomitalia.it    52140           Italy
30798        btcentralplus.com    49582  United Kingdom
68252                flyio.net    48688  United Kingdom
161811               scw.cloud    38098     Netherlands
159182            sakura.ne.jp    30383           Japan
196682    vultrusercontent.com    29061           Japan
111467   linodeusercontent.com    27771   United States
63405       ewe-ip-backbone.de    26475         Germany
171327            spectrum.com    19393   United States
197190             walmart.com    18625   United States
205061          your-server.de    17845         Germany
180562           telefonica.de    16367         Germany
70204          frontiernet.net    15043   United States
135053                 nuro.jp    14180           

In [5]:
import duckdb, tldextract, pandas as pd

df = duckdb.sql(f"""
  SELECT
    lower(json_extract_string(json, '$.hostnames[0]'))        AS host,
    lower(json_extract_string(json, '$.ssl.cert.subject.CN')) AS cn,
    json_extract_string(json, '$.location.country_name')      AS country,
    json_extract_string(json, '$.vulns')                      AS vulns
  FROM read_ndjson_objects('{f}', ignore_errors=true, maximum_object_size=134217728)
  WHERE json_extract_string(json, '$.hostnames') != '[]'
""").df()

extract = tldextract.TLDExtract(suffix_list_urls=())

def root(h):
    if not h: return None
    e = extract(h)
    return f"{e.domain}.{e.suffix}" if e.domain and e.suffix else None

def pick(cn, host):
    cand = (cn or "").lstrip("*.") or host
    return root(cand)

df["company"] = [pick(c, h) for c, h in zip(df["cn"], df["host"])]

# provider giveaway words — drop these
BADWORDS = ['isp','cloud','dns','broadband','cable','hosting','server','telecom',
            'backbone','ip-','-ip','vps','dedicated','datacenter','host','net.',
            'amazonaws','googleusercontent','azure','akamai','cloudfront']

def is_provider(d):
    return d is None or any(w in d for w in BADWORDS)

df = df[~df["company"].apply(is_provider)]

df.to_parquet("companies_raw.parquet")
print("saved:", len(df), "rows |", df["company"].nunique(), "unique companies")
print(df["company"].value_counts().head(20))

saved: 1008610 rows | 188774 unique companies
company
imperva.com                 68892
btcentralplus.com           49582
flyio.net                   48688
sakura.ne.jp                30383
vultrusercontent.com        29061
linodeusercontent.com       27771
spectrum.com                19393
walmart.com                 18625
telefonica.de               16367
nuro.jp                     14180
unifiedlayer.com            11890
bizmw.com                    9050
ovh.net                      7545
etius.jp                     6648
awsglobalaccelerator.com     6389
myvzw.com                    5807
znlc.jp                      5015
kpn.net                      4557
qwest.net                    4339
websitewelcome.com           4158
Name: count, dtype: int64


In [6]:
counts = df["company"].value_counts()
# keep companies with a believable number of servers (tune 500 if needed)
real = counts[counts <= 500].index
clean = df[df["company"].isin(real)]

clean.to_parquet("companies_raw.parquet")
print("kept:", len(clean), "rows |", clean["company"].nunique(), "companies")
print(clean["company"].value_counts().head(20))

kept: 444644 rows | 188613 companies
company
microsoft.com     500
vivozap.com.br    494
netia.com.pl      491
aws.dev           487
asus.com          486
exetel.com.au     484
fuse.net          483
mdcc-fun.de       482
as62651.net       477
3cx.us            476
mesh.ad.jp        473
synology.com      471
net-htp.de        469
virginm.net       464
level3.net        460
mrse.com.ar       458
ctm.net           458
cyber-folks.pl    457
salesforce.com    457
windows.net       457
Name: count, dtype: int64


In [7]:
import pandas as pd
df = pd.read_parquet("companies_raw.parquet")

# show a few rows that actually have vulns
has = df[df["vulns"].notna() & (df["vulns"] != "{}") & (df["vulns"] != "")]
print("rows with vulns:", len(has))
print(has[["company","country","vulns"]].head(3).to_string())

rows with vulns: 34073
                 company        country                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [9]:
import pandas as pd, json, urllib.request

# 1. download KEV once
url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
urllib.request.urlretrieve(url, "kev.json")

kev_raw = json.load(open("kev.json"))
kev = {v["cveID"]: v.get("knownRansomwareCampaignUse","Unknown")
       for v in kev_raw["vulnerabilities"]}
print("KEV entries loaded:", len(kev))

# 2. re-score, now aware of KEV
df = pd.read_parquet("companies_raw.parquet")
df = df[~df["company"].str.contains(r"\.arpa|in-addr", case=False, na=False)]
def score(vulns_text):
    if not vulns_text or vulns_text in ("{}",""): 
        return pd.Series([0.0,0.0,0,False,False])
    try: v = json.loads(vulns_text)
    except: return pd.Series([0.0,0.0,0,False,False])
    if not isinstance(v,dict) or not v: 
        return pd.Series([0.0,0.0,0,False,False])
    cves = list(v.keys())
    max_cvss = max((c.get("cvss") or 0) for c in v.values())
    max_epss = max((c.get("epss") or 0) for c in v.values())
    in_kev = any(c in kev for c in cves)
    ransom = any(kev.get(c,"") == "Known" for c in cves)
    return pd.Series([max_cvss, max_epss, len(v), in_kev, ransom])

df[["max_cvss","max_epss","cve_count","in_kev","ransomware"]] = df["vulns"].apply(score)
scored = df[df["cve_count"] > 0].copy()

company = (scored.sort_values(["in_kev","max_epss","max_cvss"], ascending=False)
           .groupby("company")
           .agg(country=("country","first"),
                max_cvss=("max_cvss","max"),
                max_epss=("max_epss","max"),
                cve_count=("cve_count","sum"),
                in_kev=("in_kev","max"),
                ransomware=("ransomware","max"))
           .reset_index())

def tier(r):
    if r["in_kev"]: return "Critical"          # being attacked now = top
    if r["max_epss"] >= 0.5: return "High"
    if r["max_epss"] >= 0.1: return "Medium"
    return "Low"

company["tier"] = company.apply(tier, axis=1)
company = company.sort_values(["in_kev","max_epss","max_cvss"], ascending=False)
company.to_parquet("companies_scored.parquet")

print(company["tier"].value_counts())
print("KEV-flagged companies:", company["in_kev"].sum())
print("Ransomware-linked:", company["ransomware"].sum())
print(company.head(15).to_string())

KEV entries loaded: 1662
tier
Critical    14239
High         5388
Low          4110
Medium       1380
Name: count, dtype: int64
KEV-flagged companies: 14239
Ransomware-linked: 3917
                    company        country  max_cvss  max_epss  cve_count  in_kev  ransomware      tier
7287            entreda.net  United States      10.0   0.94489        241    True        True  Critical
13324              lobap.ca         Canada       9.8   0.94485        824    True        True  Critical
2098   atcmultimidia.com.br         Brazil       9.8   0.94469        223    True        True  Critical
5074           contract.org   South Africa       9.8   0.94469          1    True       False  Critical
6763       eastern-tele.com    Philippines       9.8   0.94469        193    True        True  Critical
16653    ops-console.com.au      Indonesia       9.8   0.94469          1    True       False  Critical
19942       shoptour.com.br         Brazil       9.8   0.94469          1    True       Fal

In [10]:
company.to_parquet("companies_scored.parquet")
print("FINAL:", len(company), "companies saved")

FINAL: 25117 companies saved


In [11]:
import pandas as pd
df = pd.read_parquet("companies_scored.parquet")
df.to_csv("companies.csv", index=False)
print("saved", len(df), "rows")

saved 25117 rows
